# Middleware

#### Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
##### Tracking agent behavior with logging, analytics, and debugging.
##### Transforming prompts, tool selection, and output formatting.
##### Adding retries, fallbacks, and early termination logic.
##### Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["OLLAMA_MODEL"] = os.getenv("OLLAMA_MODEL")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

### Message based summarization
agent = create_agent(
    model=os.environ["OLLAMA_MODEL"],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=os.environ["OLLAMA_MODEL"],
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [3]:
## Run with thread
config = {"configurable": {"thread_id":"test-1"}}

In [8]:
# Alternative test data
questions = [
    "What is 2 + 2 ?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15 - 7",
    "What is 10*5?",
    "What is 10*5?",
    "What is 10*5?"
]

In [9]:
for q in questions:
    response = agent.invoke(
        {
            "messages": [HumanMessage(content=q)]
        }, config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user's primary goal is to practice and verify simple arithmetic calculations (addition, subtraction, multiplication, division).\n\n## SUMMARY\n\nA series of basic math problems have been solved, demonstrating successful execution across various operations:\n*   2 + 2 = 4 (Addition)\n*   10 * 5 = 50 (Multiplication)\n*   100 / 4 = 25 (Division)\n*   15 - 7 = 8 (Subtraction)\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nThe overall goal of solving arithmetic problems has been successfully demonstrated. The session is ready for any further arithmetic challenges the user wishes to propose.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f32de78a-1cce-43b4-a301-1cf9e9f7e0e9'), AIMessage(content='50', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T12:18:12.289892Z', 'done': True, 'done_reason': 'stop', 'tot

# Human in the loop

#### Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
#### High-stakes operations requiring human approval (e.g. database writes, financial transactions).
#### Compliance workflows where human oversight is mandatory.
#### Long-running conversations where human feedback guides the agent.

In [11]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by it's ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email send to {recipient} with subject '{subject}'"

agent = create_agent(
    model= os.environ["OLLAMA_MODEL"],
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool" : {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False
            }
        ),
    ],
)

In [24]:
config = {"configurable": {"thread_id": "test-approve"}}

request = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'")
        ]
    },
    config=config
)

In [25]:
request

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='801dfe85-2dc1-4c7a-9aa4-8fa40bbcb85f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T13:24:17.01086Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13862309042, 'load_duration': 3150990750, 'prompt_eval_count': 159, 'prompt_eval_duration': 597087000, 'eval_count': 294, 'eval_duration': 10106993000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'}, id='lc_run--019fe18b-7dca-7fe2-a857-0fa93c94b9ad-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'a4851724-96cf-414d-ba4e-0cdf8d725548', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 294, 'total_tokens': 453}),
  HumanMessage(content="Send email to john@tes

In [26]:
from langgraph.types import Command

if "__interrupt__" in request:
    print("Paused! Approving...")

    request = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )

    print(f"Result: {request['messages'][-1].content}")

Paused! Approving...
Result: I have sent the email to john@test.com with the subject 'Hello' and body 'How are you'.


In [27]:
request

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='801dfe85-2dc1-4c7a-9aa4-8fa40bbcb85f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T13:24:17.01086Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13862309042, 'load_duration': 3150990750, 'prompt_eval_count': 159, 'prompt_eval_duration': 597087000, 'eval_count': 294, 'eval_duration': 10106993000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'}, id='lc_run--019fe18b-7dca-7fe2-a857-0fa93c94b9ad-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you', 'recipient': 'john@test.com', 'subject': 'Hello'}, 'id': 'a4851724-96cf-414d-ba4e-0cdf8d725548', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 294, 'total_tokens': 453}),
  HumanMessage(content="Send email to john@tes